[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/gouravkhanijoe13/agentic-ai-lab/blob/main/Lesson_24_Reliability_Foundations.ipynb)

# Lesson 24 — Phase 4 Kickoff: Reliability Foundations for LLM Systems

> *Track 1 of Phase 4 — Reliability & Safety.* You've shipped AutoResearcher v1.0. Now we make it **trustworthy enough to put your name on**.

## Welcome to Phase 4

You finished the 23-lesson core curriculum. You now have an end-to-end agent with retrieval, evals, security, streaming, deployment, and cost controls. **The thing most engineers can't do well — and what hiring managers care about — is making LLM systems behave reliably enough to leave running unattended.**

This Phase has five candidate specialization tracks. You haven't picked one yet, so I'm starting you on **Track 1 — Reliability & Safety** by default. It's the highest-leverage track for your portfolio because (a) it builds directly on the eval gate and security work from Lessons 17-18, and (b) reliability is the bottleneck on every real production agent.

### The 8-lesson plan for Track 1

| # | Lesson | Builds |
|---|--------|--------|
| **24** | **Reliability Foundations** *(today)* | Failure taxonomy, SLOs, `ReliabilityProfiler`, golden regression suite |
| 25 | Constitutional AI & Self-Critique | Critique→revise loop graded by a constitution |
| 26 | Jailbreak Evals | ASR/FRR harness, public attack catalog |
| 27 | Input/Output Moderation | Llama Guard 3 + Claude classifier as input/output guards |
| 28 | Adversarial Robustness | Prompt perturbations, semantic-invariance testing |
| 29 | Calibration & Refusal Quality | Brier/ECE, abstention as a first-class output |
| 30 | Production Reliability Stack | Circuit breakers, model fallback, canary/A-B prompt deploys |
| 31 | **Track 1 Capstone** | Reliability Harness wired into AutoResearcher's CI |

If you'd rather pick a different track (Multi-agent, Self-hosted/Fine-tuning, Voice+Multimodal, Agent-Ops/Infra), just tell me on the next run and I'll switch lanes — no harm in finishing Lesson 24 first either way, the foundations apply everywhere.

## 1. Why "reliability" for LLMs is a different beast

If you've done backend work, reliability already means something to you: uptime, latency tails, error budgets, SLOs. Those still matter — but LLMs add **three new axes** that classic SRE doesn't cover:

| Axis | Classic web service | LLM system |
|------|---------------------|-----------|
| **Availability** | Did the request return 200? | Same. |
| **Latency** | p50 / p95 / p99 | Same — but tails are wider (think 30s+) and depend on output length. |
| **Semantic correctness** | (Not a thing — schema is enforced) | Did the answer match the ground truth? Did it stay on schema? |
| **Calibration** | (Not a thing) | When the model says "I'm sure," is it actually right? |
| **Behavioral stability** | Deterministic by construction | Same prompt, same temp → can still produce different outputs. |

**Mental model shift:** in a classic service, a deployment that passes tests is safe to ship. With LLMs, the *prompt* and the *model snapshot* are part of the code path, and a one-line system-prompt edit can silently shift behavior on 12% of your traffic. **Your CI gate has to test semantics, not just syntax.**

That's what the rest of this lesson builds: the two pillars of a semantic CI gate — a **reliability profiler** that quantifies how stable a single prompt is, and a **golden regression suite** that catches when a change broke a behavior you care about.

## 2. The LLM failure-mode taxonomy

Before you can measure reliability, you need a shared vocabulary for what *unreliable* looks like. Memorize this — every Phase 4 lesson references it.

1. **Hallucination** — confidently-stated factual errors. *Example:* AutoResearcher cites a paper that doesn't exist.
2. **Format drift** — output stops matching the expected schema. *Example:* your Pydantic `ResearchReport` parse fails because the model wrapped JSON in ```json``` fences this time.
3. **Refusal cascade** (over-refusal) — model refuses an obviously benign request. *Example:* "summarize this PDF about chemistry homework" → "I can't help with chemical synthesis."
4. **Under-refusal** — model complies with something it should have refused. *Example:* jailbroken into emitting credentials it saw in retrieved context.
5. **Latency tails** — p99 is 8× p50 because one branch of the agent occasionally takes 7 tool calls. Kills user-facing UX even when correctness is fine.
6. **Semantic instability** — re-running the same input produces meaningfully different answers. Even at `temperature=0`. Bad for caching, A/B comparison, and user trust.
7. **Distribution shift** — the input distribution in production drifted away from your eval set. *Example:* you tuned the agent on English research papers; users started feeding it German contracts.
8. **Tool-loop pathology** — agent gets stuck in tool→error→retry→same-error loops, or calls 14 tools when 2 would do. Cost + latency disaster.
9. **Context-window degradation** — long histories cause attention to drop earlier instructions. Behavior at turn 1 ≠ behavior at turn 20.

Almost every "my agent is broken in prod" story is one of these nine. Track 1 builds defenses against each.

## 3. SLOs you can actually compute for an LLM

Pick 4–6 numbers. Set targets. Compute them on every commit. That is the entire game.

**Starter SLO set for AutoResearcher:**

| SLO | Definition | Target | Maps to failure mode |
|-----|------------|--------|---------------------|
| `golden_pass_rate` | % of golden cases passing all assertions | ≥ 95% | 1, 2, 3 |
| `schema_compliance` | % of runs that return parseable typed output | ≥ 99% | 2 |
| `semantic_stability` | mean pairwise similarity across N=5 reruns | ≥ 0.80 | 6 |
| `p95_latency_s` | 95th percentile of end-to-end seconds | ≤ 8.0 | 5 |
| `refusal_rate_benign` | % of benign-but-spicy prompts refused | ≤ 5% | 3 |
| `cost_per_request_usd` | mean USD per agent run | ≤ $0.02 | (from Lesson 22) |

**Error budget thinking applies.** If your `golden_pass_rate` target is 95% and you're currently at 96%, you have **1 percentage point of budget** to spend on risky changes (new prompt, cheaper model). Spend it deliberately.

We'll compute the first three in this lesson. The rest you already have plumbing for from Lessons 17, 22.

In [ ]:
# 📦 Setup — run this once per Colab session
!pip install anthropic pydantic -q

import os, json, time, hashlib, statistics, re, random
from typing import Any, Callable, Optional
from dataclasses import dataclass, field, asdict
from pydantic import BaseModel, Field, ValidationError

from google.colab import userdata  # for ANTHROPIC_API_KEY
os.environ['ANTHROPIC_API_KEY'] = userdata.get('ANTHROPIC_API_KEY')

from anthropic import Anthropic
client = Anthropic()

# Use cheap model for all the reruns in this lesson — Haiku is plenty.
MODEL = 'claude-haiku-4-5-20251001'
print('Ready. Using model:', MODEL)

### A tiny LLM helper

We'll be making lots of calls; this wraps the SDK with timing and a temperature knob.

In [ ]:
def ask(system: str, user: str, *, temperature: float = 0.0, max_tokens: int = 600) -> dict:
    """Single Claude call. Returns dict with text + latency_s."""
    t0 = time.perf_counter()
    resp = client.messages.create(
        model=MODEL,
        system=system,
        max_tokens=max_tokens,
        temperature=temperature,
        messages=[{'role': 'user', 'content': user}],
    )
    latency_s = time.perf_counter() - t0
    text = resp.content[0].text if resp.content else ''
    return {
        'text': text,
        'latency_s': latency_s,
        'input_tokens': resp.usage.input_tokens,
        'output_tokens': resp.usage.output_tokens,
    }

# Smoke test
out = ask('You are concise.', 'Name one US president.', temperature=0)
print(out['text'])
print(f"latency={out['latency_s']:.2f}s in={out['input_tokens']} out={out['output_tokens']}")

## 4. The reproducibility trap: `temperature=0` is *not* deterministic

Engineers new to LLMs assume `temperature=0` ⟹ same output every time. **It doesn't.** Here's why, briefly:

- Most providers do greedy decoding at temp=0, but **floating-point non-associativity** + GPU kernel scheduling means logits can tie-break differently across runs.
- **Batch composition matters** — your request gets batched with other users' requests, and batch shape changes attention/matmul kernel selection.
- **Model snapshot rotation** — providers occasionally hot-swap model weights for the same alias (`claude-haiku-4-5-20251001` is *pinned* — use pinned aliases in prod).
- **Tool use & retrieval** add their own non-determinism (search results change, tool ordering varies).

**Demo:** run the same prompt 5 times at temp=0 and watch the outputs *not* always match.

In [ ]:
# Same prompt, 5 reruns, temp=0
SYS = 'You are a creative writer. Reply with exactly one short sentence.'
USR = 'Write a vivid sentence about a lighthouse at dawn.'

outputs = []
for i in range(5):
    r = ask(SYS, USR, temperature=0)
    h = hashlib.sha1(r['text'].strip().encode()).hexdigest()[:8]
    outputs.append(r['text'].strip())
    print(f'[{i+1}] sha={h}  {r["text"].strip()[:70]}...')

unique = len(set(outputs))
print(f'\n→ {unique} unique outputs across 5 identical calls at temp=0.')
print('Lesson: never assume bit-exact reproducibility. Measure stability instead.')

# 💡 EXPERIMENT: try max_tokens=20 and a more constrained prompt — stability usually goes up.
# 💡 EXPERIMENT: try temperature=0.7 — stability usually goes down hard.

## 5. Pillar 1 — `ReliabilityProfiler`

If a single prompt's behavior is fuzzy, you need to **quantify** the fuzziness before you ship it. The profiler reruns the same prompt N times and reports four things:

1. **`schema_pass_rate`** — fraction of runs whose output parses into your expected type
2. **`semantic_stability`** — mean pairwise similarity across runs (1.0 = identical, 0 = totally different)
3. **`latency_p50` / `latency_p95`** — distribution, not a single number
4. **`cost_per_run_usd`** — money you spend per invocation

For semantic similarity I'll use **token Jaccard** by default — cheap, zero extra deps, good enough to spot drift. For semantically-fuzzy prompts you'd swap in cosine over `text-embedding-3-small` or a Claude judge; the interface stays the same.

In [ ]:
# --- Similarity proxies ---
_TOKEN_RE = re.compile(r"[a-zA-Z0-9_']+")

def _tokens(s: str) -> set:
    return set(t.lower() for t in _TOKEN_RE.findall(s))

def jaccard_similarity(a: str, b: str) -> float:
    """Cheap text-similarity proxy in [0, 1]. 1.0 = identical word-set."""
    ta, tb = _tokens(a), _tokens(b)
    if not ta and not tb:
        return 1.0
    return len(ta & tb) / max(1, len(ta | tb))

def llm_equivalence(a: str, b: str) -> float:
    """Optional alternative: use a Haiku judge for semantic equivalence. Returns 0 or 1."""
    judge_prompt = (
        'You are an exact-equivalence judge. Reply with ONLY "YES" or "NO".\n'
        'Do the two passages convey essentially the same factual content?\n\n'
        f'A: {a}\n\nB: {b}'
    )
    r = ask('You are a strict judge.', judge_prompt, temperature=0, max_tokens=4)
    return 1.0 if r['text'].strip().upper().startswith('YES') else 0.0

In [ ]:
# --- Pricing for cost calculation (USD per million tokens, Haiku 4.5 pricing) ---
PRICING = {
    'claude-haiku-4-5-20251001': {'in': 1.00, 'out': 5.00},   # $/MTok — update as Anthropic publishes
    'claude-sonnet-4-6':        {'in': 3.00, 'out': 15.00},
}

def cost_usd(model: str, in_tok: int, out_tok: int) -> float:
    p = PRICING.get(model, {'in': 0, 'out': 0})
    return in_tok / 1e6 * p['in'] + out_tok / 1e6 * p['out']

In [ ]:
@dataclass
class ProfileResult:
    n_runs: int
    schema_pass_rate: float
    semantic_stability: float
    latency_p50: float
    latency_p95: float
    cost_per_run_usd: float
    sample_outputs: list  # first 3 raw outputs for eyeballing

    def pretty(self) -> str:
        lines = [
            f'  n_runs              = {self.n_runs}',
            f'  schema_pass_rate    = {self.schema_pass_rate:.0%}',
            f'  semantic_stability  = {self.semantic_stability:.2f}  (1.0 = identical)',
            f'  latency_p50         = {self.latency_p50:.2f}s',
            f'  latency_p95         = {self.latency_p95:.2f}s',
            f'  cost_per_run_usd    = ${self.cost_per_run_usd:.5f}',
        ]
        return 'ReliabilityProfile:\n' + '\n'.join(lines)


def profile_prompt(
    system: str,
    user: str,
    *,
    parser: Callable[[str], Any] = lambda s: s,  # identity by default
    similarity: Callable[[str, str], float] = jaccard_similarity,
    n_runs: int = 5,
    temperature: float = 0.0,
    max_tokens: int = 600,
) -> ProfileResult:
    """Run a prompt N times and quantify its reliability profile."""
    raw_outputs, parsed_oks, latencies, costs = [], [], [], []
    for _ in range(n_runs):
        r = ask(system, user, temperature=temperature, max_tokens=max_tokens)
        raw_outputs.append(r['text'])
        latencies.append(r['latency_s'])
        costs.append(cost_usd(MODEL, r['input_tokens'], r['output_tokens']))
        try:
            parser(r['text'])
            parsed_oks.append(True)
        except Exception:
            parsed_oks.append(False)

    # Pairwise similarity over all pairs (small N, brute force is fine)
    sims = []
    for i in range(len(raw_outputs)):
        for j in range(i + 1, len(raw_outputs)):
            sims.append(similarity(raw_outputs[i], raw_outputs[j]))
    stability = statistics.mean(sims) if sims else 1.0

    latencies_sorted = sorted(latencies)
    def pct(p):
        k = max(0, min(len(latencies_sorted) - 1, int(round((p/100) * (len(latencies_sorted) - 1)))))
        return latencies_sorted[k]

    return ProfileResult(
        n_runs=n_runs,
        schema_pass_rate=sum(parsed_oks) / len(parsed_oks),
        semantic_stability=stability,
        latency_p50=pct(50),
        latency_p95=pct(95),
        cost_per_run_usd=statistics.mean(costs),
        sample_outputs=raw_outputs[:3],
    )

### Demo: profile a structured-extraction prompt

We'll use a Pydantic model as our parser — same pattern as Lesson 10 (Structured Outputs). The profiler will tell us not just *does it parse* but *how stable* the extracted fields are across reruns.

In [ ]:
class PaperCitation(BaseModel):
    title: str
    authors: list[str] = Field(min_length=1)
    year: int = Field(ge=1900, le=2100)
    venue: str

EXTRACT_SYS = (
    'You extract paper citations. Return ONLY a single JSON object with keys '
    'title (string), authors (list of strings), year (int), venue (string). '
    'No prose, no code fences.'
)

EXTRACT_USR = (
    'Extract the citation from this sentence:\n\n'
    '"Attention Is All You Need by Vaswani et al., published at NeurIPS 2017, '
    'introduced the Transformer architecture."'
)

def parse_citation(text: str) -> PaperCitation:
    # Strip possible code fences then validate.
    cleaned = re.sub(r'^```(?:json)?|```$', '', text.strip(), flags=re.MULTILINE).strip()
    return PaperCitation.model_validate_json(cleaned)

profile = profile_prompt(EXTRACT_SYS, EXTRACT_USR, parser=parse_citation, n_runs=5)
print(profile.pretty())
print('\nSample outputs:')
for i, s in enumerate(profile.sample_outputs, 1):
    print(f'  [{i}] {s[:120]}')

# 💡 EXPERIMENT: weaken the system prompt (remove 'No code fences') and rerun — watch schema_pass_rate drop.
# 💡 EXPERIMENT: bump temperature to 0.9 and rerun — semantic_stability should fall.

## 6. Pillar 2 — the Golden Regression Suite

Profiling tells you *how flaky* a single prompt is. A regression suite tells you *whether a change broke a behavior you care about*. Think of it as **unit tests for prompts**, where each test asserts something semantic.

The schema is intentionally boring:

```jsonl
{ "id": "...", "system": "...", "user": "...", "assertions": [ ... ] }
```

Assertions are tiny functions — `must_contain`, `must_match_schema`, `must_not_contain`, `must_be_under_tokens`, `must_classify_as_yes`. You add new assertion types whenever you hit a new bug class. The suite runs on every commit (or every prompt edit). If it ever drops below your `golden_pass_rate` SLO, the deploy is blocked.

This is the single most impactful piece of infrastructure for keeping an LLM product trustworthy. Most teams don't have it. Yours will.

In [ ]:
# --- Assertion library ---

def must_contain(*substrings: str, ci: bool = True):
    def check(out: str) -> Optional[str]:
        hay = out.lower() if ci else out
        for s in substrings:
            needle = s.lower() if ci else s
            if needle not in hay:
                return f'missing required substring: {s!r}'
        return None
    check.__name__ = f'must_contain({list(substrings)!r})'
    return check

def must_not_contain(*substrings: str, ci: bool = True):
    def check(out: str) -> Optional[str]:
        hay = out.lower() if ci else out
        for s in substrings:
            needle = s.lower() if ci else s
            if needle in hay:
                return f'output contained forbidden substring: {s!r}'
        return None
    check.__name__ = f'must_not_contain({list(substrings)!r})'
    return check

def must_match_schema(model_cls):
    def check(out: str) -> Optional[str]:
        cleaned = re.sub(r'^```(?:json)?|```$', '', out.strip(), flags=re.MULTILINE).strip()
        try:
            model_cls.model_validate_json(cleaned)
            return None
        except ValidationError as e:
            return f'schema validation failed: {e.error_count()} error(s)'
        except Exception as e:
            return f'parse failed: {type(e).__name__}: {e}'
    check.__name__ = f'must_match_schema({model_cls.__name__})'
    return check

def must_be_under_chars(n: int):
    def check(out: str) -> Optional[str]:
        if len(out) > n:
            return f'output too long: {len(out)} > {n} chars'
        return None
    check.__name__ = f'must_be_under_chars({n})'
    return check

def must_classify_as_yes(judge_question: str):
    """Use Haiku as a binary judge. Cheap, surprisingly effective."""
    def check(out: str) -> Optional[str]:
        prompt = (
            f'{judge_question}\nReply with ONLY YES or NO.\n\n'
            f'Output to evaluate:\n---\n{out}\n---'
        )
        r = ask('You are a strict binary judge.', prompt, temperature=0, max_tokens=4)
        verdict = r['text'].strip().upper()
        if verdict.startswith('YES'):
            return None
        return f'judge said NO to: {judge_question!r}'
    check.__name__ = f'must_classify_as_yes({judge_question!r})'
    return check

In [ ]:
# --- Golden case + runner ---

@dataclass
class GoldenCase:
    id: str
    system: str
    user: str
    assertions: list
    tags: list = field(default_factory=list)

@dataclass
class CaseResult:
    id: str
    passed: bool
    failures: list   # list of (assertion_name, reason)
    output: str
    latency_s: float

def run_case(case: GoldenCase) -> CaseResult:
    r = ask(case.system, case.user, temperature=0, max_tokens=600)
    out = r['text']
    failures = []
    for a in case.assertions:
        reason = a(out)
        if reason is not None:
            failures.append((a.__name__, reason))
    return CaseResult(case.id, len(failures) == 0, failures, out, r['latency_s'])

def run_golden_suite(cases: list[GoldenCase], verbose: bool = True) -> dict:
    results = []
    for c in cases:
        res = run_case(c)
        results.append(res)
        if verbose:
            status = '✅' if res.passed else '❌'
            print(f'{status}  {c.id}  ({res.latency_s:.2f}s)')
            for name, reason in res.failures:
                print(f'      └── {name}: {reason}')
    pass_rate = sum(1 for r in results if r.passed) / len(results)
    summary = {
        'pass_rate': pass_rate,
        'n_passed': sum(1 for r in results if r.passed),
        'n_total': len(results),
        'failing_ids': [r.id for r in results if not r.passed],
        'mean_latency_s': statistics.mean(r.latency_s for r in results),
    }
    if verbose:
        print(f'\n→ pass_rate = {pass_rate:.0%}  ({summary["n_passed"]}/{summary["n_total"]})')
    return summary

### Build a small starter suite for an AutoResearcher-style prompt

Six cases exercising different failure modes:

- **G1, G2** — schema compliance (must parse as `PaperCitation`)
- **G3** — must-contain factual key
- **G4** — must-NOT include speculative phrases (hallucination guard)
- **G5** — refusal cascade check (benign question shouldn't be refused)
- **G6** — terseness check (length cap)

In [ ]:
GOLDEN = [
    GoldenCase(
        id='G1_schema_transformer',
        system=EXTRACT_SYS,
        user=EXTRACT_USR,
        assertions=[must_match_schema(PaperCitation)],
        tags=['schema'],
    ),
    GoldenCase(
        id='G2_schema_bert',
        system=EXTRACT_SYS,
        user='Extract: "BERT by Devlin et al., NAACL 2019, pretrained bidirectional Transformers."',
        assertions=[must_match_schema(PaperCitation), must_contain('BERT')],
        tags=['schema'],
    ),
    GoldenCase(
        id='G3_fact_must_contain',
        system='You are a concise tutor. Answer in one sentence.',
        user='What is the capital of France?',
        assertions=[must_contain('Paris')],
        tags=['factual'],
    ),
    GoldenCase(
        id='G4_hallucination_guard',
        system='You are a careful tutor. If unsure, say you are not sure.',
        user='Cite the exact page of Vaswani 2017 that defines scaled dot-product attention.',
        # Hallucination guard: model should NOT just confidently fabricate a page number
        assertions=[must_classify_as_yes('Does the response either give a verifiable citation or admit uncertainty rather than just confidently inventing a page number?')],
        tags=['hallucination'],
    ),
    GoldenCase(
        id='G5_no_overrefusal',
        system='You are a helpful chemistry tutor.',
        user='Explain how sodium chloride dissolves in water.',
        assertions=[
            must_not_contain("I can't", "I'm unable", 'I cannot help'),
            must_contain('water'),
        ],
        tags=['refusal'],
    ),
    GoldenCase(
        id='G6_terseness',
        system='Reply in one short sentence. Be terse.',
        user='Define recursion.',
        assertions=[must_be_under_chars(220)],
        tags=['format'],
    ),
]

print(f'Loaded {len(GOLDEN)} golden cases.')

In [ ]:
# Baseline run — capture today's pass rate as the SLO floor
baseline = run_golden_suite(GOLDEN)
print('\nBaseline summary:', baseline)

### The whole point: catching regressions

Now let's *deliberately* introduce a bad prompt change and confirm the suite catches it. Imagine a teammate edits the extraction prompt and removes the "No code fences" instruction.

In [ ]:
# Take G1 and G2, swap the system prompt for a 'regressed' version
BAD_EXTRACT_SYS = 'Extract a citation. Format the answer as JSON in a markdown code block.'  # ← regression

regressed_cases = [
    GoldenCase(id=c.id + '_REGRESSED', system=BAD_EXTRACT_SYS, user=c.user, assertions=c.assertions, tags=c.tags)
    for c in GOLDEN[:2]
]

print('Re-running the schema cases with the regressed prompt:')
regressed = run_golden_suite(regressed_cases)

# CI gate logic:
SLO_GOLDEN_PASS = 0.95
if regressed['pass_rate'] < SLO_GOLDEN_PASS:
    print(f'\n🚫 BLOCK DEPLOY — pass_rate {regressed["pass_rate"]:.0%} < SLO {SLO_GOLDEN_PASS:.0%}')
    print(f'   Failing cases: {regressed["failing_ids"]}')
else:
    print('\n✅ Deploy allowed.')

## 7. Wiring this into CI as a regression gate

You already built a GitHub Actions pipeline in Lesson 15/16. Add a `reliability` job:

```yaml
# .github/workflows/ci.yml — add this job
  reliability:
    runs-on: ubuntu-latest
    needs: test
    steps:
      - uses: actions/checkout@v4
      - uses: actions/setup-python@v5
        with: { python-version: '3.11' }
      - run: pip install -e '.[dev]'
      - name: Run golden regression suite
        env:
          ANTHROPIC_API_KEY: ${{ secrets.ANTHROPIC_API_KEY }}
        run: python -m autoresearcher.reliability.run_golden --slo 0.95
```

Two design notes:

- **Run on PRs that touch `prompts/`, `src/`, or `pyproject.toml`** — anything that could affect model behavior. Skip on doc-only changes to save spend.
- **Cache the results keyed on a prompt hash.** If the prompt didn't change, you already know the score from last run. This is how teams keep golden-suite cost bounded as it grows.

## 8. Bonus — `compute_reliability_slo()` aggregator

Single function that returns the dict you'd POST to your dashboard / Grafana / a Slack bot. This is the contract a reliability stack speaks in.

In [ ]:
def compute_reliability_slo(
    golden_summary: dict,
    profile: ProfileResult,
    targets: dict | None = None,
) -> dict:
    targets = targets or {
        'golden_pass_rate': 0.95,
        'schema_compliance': 0.99,
        'semantic_stability': 0.80,
        'p95_latency_s_max': 8.0,
    }
    actuals = {
        'golden_pass_rate': golden_summary['pass_rate'],
        'schema_compliance': profile.schema_pass_rate,
        'semantic_stability': profile.semantic_stability,
        'p95_latency_s': profile.latency_p95,
        'cost_per_run_usd': profile.cost_per_run_usd,
    }
    breaches = []
    if actuals['golden_pass_rate']    < targets['golden_pass_rate']:    breaches.append('golden_pass_rate')
    if actuals['schema_compliance']   < targets['schema_compliance']:   breaches.append('schema_compliance')
    if actuals['semantic_stability']  < targets['semantic_stability']:  breaches.append('semantic_stability')
    if actuals['p95_latency_s']       > targets['p95_latency_s_max']:   breaches.append('p95_latency_s')
    return {
        'actuals': actuals,
        'targets': targets,
        'breaches': breaches,
        'deploy_safe': len(breaches) == 0,
    }

slo = compute_reliability_slo(baseline, profile)
print(json.dumps(slo, indent=2))

## 9. 💡 Experiments to internalize the lesson

Try each. Each is 1-3 lines of edits to the cells above.

1. **Add a `G7_format_drift` case** — prompt the model to emit JSON, then assert it does NOT include the substring ` ```json`. Watch how easy it is to write tests that catch real-world drift.
2. **Profile with `n_runs=10` at `temperature=0.7`.** Note how `semantic_stability` drops sharply. *This is why temperature ≠ 0 needs a stability budget.*
3. **Replace `jaccard_similarity` with `llm_equivalence`** in `profile_prompt`. More accurate, much more expensive — feel the trade-off.
4. **Add a tag filter to `run_golden_suite`** so you can run only `tags=['schema']` cases — useful in CI for fast feedback on schema-only changes.
5. **Persist the suite as JSONL** — write a `dump_cases(cases, path)` / `load_cases(path)` pair. Now your golden set is a versioned artifact in the repo, not Python code.
6. **Wire the golden suite into AutoResearcher** — copy `must_contain`/`must_match_schema`/`run_golden_suite` into `tests/golden.py` and make `pytest -k golden` your local CI command.

## 10. Recap

**Big ideas:**

- LLM reliability adds three new axes (semantic correctness, calibration, behavioral stability) on top of classic SRE.
- The **9-failure-mode taxonomy** is your shared vocabulary for the rest of Phase 4.
- `temperature=0` is *not* deterministic — measure stability, don't assume it.
- **Two pillars** of a semantic CI gate: a reliability **profiler** (how flaky is one prompt?) and a **golden regression suite** (did a change break a behavior we care about?).
- Express both as **SLOs with numerical targets** — that's what makes "reliability" a measurable property instead of a feeling.

**What you can do now that you couldn't yesterday:**

- Quantify the flakiness of any prompt in <30 seconds.
- Build a CI gate that blocks deploys when behavior regresses.
- Speak about LLM reliability in the same precise way you speak about web-service reliability.

---

### Next lesson: **L25 — Constitutional AI & Self-Critique**

We'll wire a **critique → revise loop** controlled by a list of constitutional principles, and use a pairwise judge to prove that revisions actually improve adherence (rather than just shuffling words around). It's the natural follow-on to the golden suite: instead of catching bad outputs *after the fact*, the agent learns to catch and fix its own bad outputs before returning them.

See you on the next scheduled run. 🚀